In [1]:
import pandas as pd
df_feat = pd.read_pickle("dataset_features_v1.pkl")

In [2]:
import json
print(json.dumps(df_feat['career_history'][0][0], indent=2))

{
  "company": "Mindtree",
  "title": "Backend Engineer",
  "start_date": "2024-03-08",
  "end_date": null,
  "duration_months": 27,
  "is_current": true,
  "industry": "IT Services",
  "company_size": "10001+",
  "description": "Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure."
}


In [3]:
print(json.dumps(df_feat['skills'][0][0], indent=2))

{
  "name": "Tailwind",
  "proficiency": "intermediate",
  "endorsements": 3,
  "duration_months": 13
}


In [4]:
print(json.dumps(df_feat['education'][0][0], indent=2))

{
  "institution": "Lovely Professional University",
  "degree": "B.E.",
  "field_of_study": "Computer Science",
  "start_year": 2017,
  "end_year": 2020,
  "grade": "8.24 CGPA",
  "tier": "tier_3"
}


In [5]:
df_feat['current_company_tenure_months'] = df_feat['career_history'].apply(
    lambda x: next(
        (job['duration_months'] for job in x if job['is_current']),
        0
    )
)

In [6]:
df_feat['total_career_months'] = df_feat['career_history'].apply(
    lambda x: sum(job['duration_months'] for job in x)
)

In [7]:
df_feat['job_switch_count'] = df_feat['career_history'].apply(
    lambda x: max(len(x)-1,0)
)

In [9]:
df_feat['avg_job_duration_months'] = (
    df_feat['total_career_months']
    /
    df_feat['num_jobs']
)

In [10]:
df_feat['avg_skill_duration_months'] = df_feat['skills'].apply(
    lambda x:
    sum(skill['duration_months'] for skill in x)/len(x)
    if len(x)>0 else 0
)

In [11]:
df_feat['avg_skill_endorsements'] = df_feat['skills'].apply(
    lambda x:
    sum(skill['endorsements'] for skill in x)/len(x)
    if len(x)>0 else 0
)

In [12]:
tier_map = {
    'tier_1':1,
    'tier_2':2,
    'tier_3':3
}

df_feat['best_education_tier'] = df_feat['education'].apply(
    lambda x:
    min(
        [tier_map.get(ed['tier'],4) for ed in x],
        default=4
    )
)

In [13]:
degree_priority = {
    'PhD':4,
    'M.Tech':3,
    'M.E.':3,
    'M.S.':3,
    'B.Tech':2,
    'B.E.':2,
    'B.Sc':1
}

df_feat['highest_degree_score'] = df_feat['education'].apply(
    lambda x:
    max(
        [degree_priority.get(ed['degree'],0) for ed in x],
        default=0
    )
)

In [14]:
df_feat['current_role_text'] = (
    df_feat['profile_current_title']
    + " "
    + df_feat['profile_current_industry']
)

In [15]:
df_feat.to_pickle("dataset_features_v2.pkl")